In [1]:
# Set Up Library
# Library untuk PATH
from pathlib import Path

# Library FFMPEG dan progress bar (folder and files management)
import subprocess
from tqdm import tqdm
import shutil
import os

# Library Mediapipe Blazepose
import mediapipe as mp
from mediapipe.tasks import python
from mediapipe.tasks.python import vision

# Library OpenCV-Python
import cv2 as cv

# Library base
import numpy as np

### **Data Pre-Processing for Self-Recorded Dataset**

**1: Video Data Standardization**

In [4]:
# Read PATH self_record_dataset
data_dir = Path('self_record_dataset')

classes = [d for d in data_dir.iterdir() if d.is_dir()]
print(classes)

[WindowsPath('self_record_dataset/jumping_jacks'), WindowsPath('self_record_dataset/knee_push_up'), WindowsPath('self_record_dataset/mountain_climber'), WindowsPath('self_record_dataset/sit_up'), WindowsPath('self_record_dataset/squat')]


In [5]:
# Memeriksa jumlah video dengan ekstensi selain .mp4 dan menyimpan yang .mp4 dalam array
video_files = []
non_mp4_files = []

for file in data_dir.rglob("*"):
    if file.is_file():
        if file.suffix.lower() == ".mp4":
            video_files.append(file)      
        else:
            non_mp4_files.append(file)

if non_mp4_files:
    print("Terdapat file dengan ekstensi selain .mp4")
    for f in non_mp4_files:
        print(f)
else:
    print("Semua file memiliki ekstensi .mp4")

print(f"Total video dataset: {len(video_files)}")

Semua file memiliki ekstensi .mp4
Total video dataset: 1360


In [9]:
# Setup output folder hasil standardisasi data
output_dir = Path('self_record_dataset_standardized')
output_dir.mkdir(exist_ok=True)

In [12]:
# Konversi video ke 30 fps dengan FFMPEG
def run(cmd):
    result = subprocess.run(cmd, stdout=subprocess.PIPE, stderr=subprocess.PIPE)
    return result.returncode == 0

success_count = 0
fail_count = 0

for cls_folder in data_dir.iterdir():
    if not cls_folder.is_dir():
        continue

    out_cls = output_dir / cls_folder.name
    out_cls.mkdir(parents=True, exist_ok=True)

    for video in cls_folder.iterdir():
        if video.suffix.lower() != ".mp4":
            continue

        output_file = out_cls / (video.stem + ".mp4")

        print(f"\nMengkonversi video: {video.name}")

        # Attempt 1
        cmd1 = [
            "ffmpeg", "-y",
            "-i", str(video),
            "-vf", "fps=30",
            "-c:v", "libx264",
            "-preset", "fast",
            "-crf", "23",
            "-c:a", "aac",
            str(output_file)
        ]

        success = run(cmd1)

        # Attempt 2 (fallback kalau gagal)
        if not success:
            print("Gagal, coba fallback...")

            cmd2 = [
                "ffmpeg", "-y",
                "-i", str(video),
                "-r", "30",
                "-c:v", "libx264",
                "-c:a", "aac",
                str(output_file)
            ]

            success = run(cmd2)

        if success and output_file.exists():
            print("Berhasil")
            success_count += 1
        else:
            print("Gagal total")
            fail_count += 1

print(f"Jumlah video berhasil dikonversi: {success_count}")
print(f"Jumlah video gagal: {fail_count}")
print(f"Total diproses: {success_count + fail_count}")


Mengkonversi video: JumJacks_Velo10_00000143.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000203.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000265.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000326.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000392.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000451.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000514.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000577.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000640.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000704.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000771.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000837.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000902.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00000964.mp4
Berhasil

Mengkonversi video: JumJacks_Velo10_00001025.mp4
Berhasil

Mengkonversi video: JumJacks_Velo11_00000302.mp4
Berhasil

Mengkonversi video: JumJacks_Velo11_00000395.mp4
Berhas

**2: Pose-Based Data Extraction**

In [2]:
# Setup Mediapipe Blazepose Full .task
mediapipe_path = "pose_landmarker_full.task"

def create_landmarker():
  base_options = python.BaseOptions(model_asset_path=mediapipe_path)

  options = vision.PoseLandmarkerOptions(
        base_options=base_options,
        running_mode=vision.RunningMode.VIDEO,
        num_poses=1
    )
  
  return vision.PoseLandmarker.create_from_options(options)

In [32]:
# Read PATH self_record_dataset_standardized
data_cleaned_dir = Path('self_record_dataset_standardized')

classes = [d for d in data_cleaned_dir.iterdir() if d.is_dir()]
print(classes)

[WindowsPath('self_record_dataset_standardized/jumping_jacks'), WindowsPath('self_record_dataset_standardized/knee_push_up'), WindowsPath('self_record_dataset_standardized/mountain_climber'), WindowsPath('self_record_dataset_standardized/sit_up'), WindowsPath('self_record_dataset_standardized/squat')]


In [33]:
# Fungsi ekstraksi pose dari frame video yang sudah distandardisasi
def extract_pose_from_video(video_path, pose_landmarker):
  capture = cv.VideoCapture(str(video_path))

  fps = capture.get(cv.CAP_PROP_FPS)
  if not fps or fps != fps:
    fps = 30
  
  frame_idx = 0
  prev_timestamp = -1

  frames = []

  while True:
    ret, frame = capture.read()
    if not ret:
      break

    timestamps_ms = int(frame_idx * (1000.0 / fps))
    frame_idx += 1

    frame_rgb = cv.cvtColor(frame, cv.COLOR_BGR2RGB)

    mp_image = mp.Image(
      image_format=mp.ImageFormat.SRGB, 
      data=frame_rgb)

    result = pose_landmarker.detect_for_video(mp_image, timestamps_ms)

    if result.pose_landmarks:
      landmarks = result.pose_landmarks[0]
      frames.append([[lm.x, lm.y, lm.z] for lm in landmarks])
    else:
      frames.append(np.zeros((33, 3)))

  capture.release()

  return np.array(frames), fps  

In [25]:
# Loop fungsi ekstraksi pose ke semua video
seq_output_dir = Path('pose_sequences_self')
seq_output_dir.mkdir(exist_ok=True)

for cls_folder in data_cleaned_dir.iterdir():
  if not cls_folder.is_dir():
    continue

  out_cls = seq_output_dir / cls_folder.name
  out_cls.mkdir(parents=True, exist_ok=True)

  for video_path in cls_folder.glob("*.mp4"):
    print(f"Memproses output sekuens: {video_path.name}")

    pose_landmarker = create_landmarker()
    
    seq, fps = extract_pose_from_video(video_path, pose_landmarker)

    np.save(out_cls / f"{video_path.stem}.npy", seq)

Memproses output sekuens: JumJacks_Velo10_00000143.mp4
Memproses output sekuens: JumJacks_Velo10_00000203.mp4
Memproses output sekuens: JumJacks_Velo10_00000265.mp4
Memproses output sekuens: JumJacks_Velo10_00000326.mp4
Memproses output sekuens: JumJacks_Velo10_00000392.mp4
Memproses output sekuens: JumJacks_Velo10_00000451.mp4
Memproses output sekuens: JumJacks_Velo10_00000514.mp4
Memproses output sekuens: JumJacks_Velo10_00000577.mp4
Memproses output sekuens: JumJacks_Velo10_00000640.mp4
Memproses output sekuens: JumJacks_Velo10_00000704.mp4
Memproses output sekuens: JumJacks_Velo10_00000771.mp4
Memproses output sekuens: JumJacks_Velo10_00000837.mp4
Memproses output sekuens: JumJacks_Velo10_00000902.mp4
Memproses output sekuens: JumJacks_Velo10_00000964.mp4
Memproses output sekuens: JumJacks_Velo10_00001025.mp4
Memproses output sekuens: JumJacks_Velo11_00000302.mp4
Memproses output sekuens: JumJacks_Velo11_00000395.mp4
Memproses output sekuens: JumJacks_Velo11_00000481.mp4
Memproses 

**3: Sequence Interpolation**

Untuk sekuens pose yang memiliki frame kosong pada threshold 5% - 25%

In [6]:
# Fungsi untuk mengecek frame kosong dari sebuah video
def check_sequence_quality(seq):
  total_frames = len(seq)
  zero_frames = np.sum(np.all(seq == 0, axis=(1,2)))
  ratio = zero_frames / total_frames

  return ratio

In [7]:
# Fungsi untuk menginterpolasi frame kosong
def fill_zero_frames(seq):
    seq = seq.copy()
    T = len(seq)
    zero_mask = np.all(seq == 0, axis=(1, 2))
    valid_idx = np.where(~zero_mask)[0]

    if len(valid_idx) == 0:
        return seq

    for i in np.where(zero_mask)[0]:
        before = valid_idx[valid_idx < i]
        after = valid_idx[valid_idx > i]

        if len(before) > 0 and len(after) > 0:
            i0, i1 = before[-1], after[0]
            alpha = (i - i0) / (i1 - i0)
            seq[i] = (1 - alpha) * seq[i0] + alpha * seq[i1]
        elif len(before) > 0:
            seq[i] = seq[before[-1]]
        else:
            seq[i] = seq[after[0]]
    return seq 

In [9]:
# Fungsi untuk memproses interpolasi (Linear Interpolation)
def interpolate_zero_frames(seq):
  ratio = check_sequence_quality(seq)

  if ratio < 0.05:
    return seq
  elif ratio < 0.25:
    print(f"Interpolasi frame kosong dengan persentase: {ratio:.2%}")
    return fill_zero_frames(seq)
  else:
    return None

In [10]:
# Read PATH pose_sequences untuk data self recorded
pose_seq = Path('pose_sequences_self')

classes = [d for d in pose_seq.iterdir() if d.is_dir()]
print(classes)

[WindowsPath('pose_sequences_self/jumping_jacks'), WindowsPath('pose_sequences_self/knee_push_up'), WindowsPath('pose_sequences_self/mountain_climber'), WindowsPath('pose_sequences_self/sit_up'), WindowsPath('pose_sequences_self/squat')]


In [11]:
# Apply fungsi interpolasi pada frame kosong dari video
zero_frame_exist = False

for file in pose_seq.rglob("*.npy"):
  seq = np.load(file)

  if np.any(np.all(seq == 0, axis=(1, 2))):
    zero_frame_exist = True

  seq_clean = interpolate_zero_frames(seq)

  if seq_clean is None:
    print(f"Drop sekuens pose dari video: {file.name}")
    os.remove(file)
    continue

  np.save(file, seq_clean)

if not zero_frame_exist:
  print("Tidak ada frame video yang kosong (zero)")


Interpolasi frame kosong dengan persentase: 9.76%
Interpolasi frame kosong dengan persentase: 11.36%
Interpolasi frame kosong dengan persentase: 7.14%
Interpolasi frame kosong dengan persentase: 18.39%
Drop sekuens pose dari video: KneePushUp_Velo16_00108000.npy
Drop sekuens pose dari video: KneePushUp_Velo16_00108098.npy
Interpolasi frame kosong dengan persentase: 20.00%
Drop sekuens pose dari video: KneePushUp_Velo16_00108351.npy
Interpolasi frame kosong dengan persentase: 21.93%
Interpolasi frame kosong dengan persentase: 5.88%
Drop sekuens pose dari video: KneePushUp_Velo16_00108664.npy
Drop sekuens pose dari video: KneePushUp_Velo16_00108757.npy
Drop sekuens pose dari video: KneePushUp_Velo16_00108861.npy
Interpolasi frame kosong dengan persentase: 14.12%
Drop sekuens pose dari video: KneePushUp_Velo16_00109046.npy
Drop sekuens pose dari video: KneePushUp_Velo16_00109141.npy
Drop sekuens pose dari video: KneePushUp_Velo16_00109234.npy
Interpolasi frame kosong dengan persentase: 22